# Notebook 03 — Importação e controle de qualidade


In [1]:
import numpy as np
import pandas as pd
rng=np.random.default_rng(42)
n=250
w=np.linspace(350,2500,180)
X=np.exp(-((w-800)/450)**2)[:,None]*0.15 + rng.normal(0,0.01,(180,n))
X=np.clip(X.T + rng.normal(0,0.03,(n,1)),0,1)
y_clay=np.clip(10+65*np.mean(X[:,:60],axis=1)+rng.normal(0,3,n),0,100)
y_sand=np.clip(80-y_clay+rng.normal(0,4,n),0,100)
y_silt=np.clip(100-y_clay-y_sand+rng.normal(0,2,n),0,100)
df=pd.DataFrame(X,columns=[f'wl_{int(x)}' for x in w])
df['clay']=y_clay; df['sand']=y_sand; df['silt']=y_silt
df.head()


,wl_350,wl_362,wl_374,wl_386,wl_398,wl_410,wl_422,wl_434,wl_446,wl_458,...,wl_2427,wl_2439,wl_2451,wl_2463,wl_2475,wl_2487,wl_2500,clay,sand,silt
0,0.074173,0.070428,0.090807,0.080411,0.082895,0.096409,0.084372,0.085632,0.092235,0.089793,...,0.011240,0.014368,0.024595,0.015944,0.007461,0.016750,0.004048,21.523009,60.085258,15.253592
1,0.022244,0.039044,0.047639,0.038822,0.037712,0.063421,0.045306,0.048705,0.051613,0.060832,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,14.530662,66.464734,20.627395
2,0.086103,0.098870,0.077447,0.077618,0.086815,0.086239,0.110759,0.094286,0.108567,0.114383,...,0.028689,0.034281,0.004979,0.018359,0.024827,0.012831,0.038300,16.598195,62.622159,23.007693
3,0.045752,0.029463,0.027365,0.066005,0.055047,0.054976,0.058558,0.052941,0.064494,0.041567,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,18.895646,63.839246,19.010655
4,0.055887,0.075929,0.051795,0.102419,0.087789,0.083753,0.092189,0.108640,0.086978,0.114769,...,0.025290,0.012328,0.020855,0.019979,0.014737,0.031287,0.023536,21.842301,47.181672,30.726665


In [2]:
print('Dimensões:',df.shape)
print('Ausentes:',df.isna().sum().sum())
print('Duplicatas:',df.duplicated().sum())
df.describe().T.tail(3)


Dimensões: (250, 183)
Ausentes: 0
Duplicatas: 0


,count,mean,std,min,25%,50%,75%,max
clay,250.0,18.086722,3.529206,7.003843,15.706269,17.926191,20.734268,27.631107
sand,250.0,61.326028,5.383077,42.836407,57.972188,60.946129,64.788556,80.020241
silt,250.0,20.456831,4.305565,8.884677,17.651470,20.278952,23.074128,31.265716


# EXEMPLO DE IMPORTAÇÃO E CONTROLE DE QUALIDADE DOS DADOS ESPECTRAIS

In [4]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

# Arquivos brutos disponíveis no projeto.
DATA_DIR = Path(r"C:\Users\PC\Desktop\RESET")
OUTPUT_DIR = Path("outputs_importacao_qc")
FILES = {
    "MIR": DATA_DIR / "raw_mir_data_Saturin.csv",
    "NIR": DATA_DIR / "raw_nir_data_Saturin.csv",
}
TARGET_COLUMNS = [
    "Sand_gkg", "Silt_gkg", "Clay_gkg", "Density_kgdm3",
    "C_gkg", "OM_gkg", "P_mgkg", "pH_H2O",
]


def is_wavelength_column(column_name):
    """Identifica bandas espectrais representadas por números de onda."""
    try:
        wavelength = float(str(column_name).strip().replace(",", "."))
    except (TypeError, ValueError):
        return False
    return 300 <= wavelength <= 4000


def read_csv_with_encoding(file_path):
    """Lê CSVs que podem usar UTF-8, Windows-1252 ou Latin-1."""
    encodings = ("utf-8", "cp1252", "latin1")
    errors = []
    for encoding in encodings:
        try:
            return pd.read_csv(file_path, encoding=encoding, low_memory=False), encoding
        except UnicodeDecodeError as error:
            errors.append(f"{encoding}: {error}")
    raise UnicodeDecodeError(
        "csv", b"", 0,
        1,
        "Não foi possível decodificar o arquivo. Tentativas: " + "; ".join(errors),
    )


def import_and_qc(file_path, sensor_name):
    if not file_path.exists():
        raise FileNotFoundError(f"Arquivo {sensor_name} não encontrado: {file_path}")

    raw_data, encoding_used = read_csv_with_encoding(file_path)
    raw_data.columns = [str(column).strip() for column in raw_data.columns]

    # Remove apenas colunas auxiliares sem nome e totalmente vazias.
    unnamed_columns = [
        column for column in raw_data.columns
        if column.lower().startswith("unnamed")
    ]
    data = raw_data.drop(columns=unnamed_columns, errors="ignore")
    empty_columns = data.columns[data.isna().all()].tolist()
    data = data.drop(columns=empty_columns)

    spectral_columns = [
        column for column in data.columns if is_wavelength_column(column)
    ]

    numeric_data = data.apply(pd.to_numeric, errors="coerce")
    non_numeric_counts = {
        column: int(data[column].notna().sum() - numeric_data[column].notna().sum())
        for column in spectral_columns
    }
    non_numeric_counts = {
        column: count for column, count in non_numeric_counts.items() if count > 0
    }

    duplicate_rows = int(data.duplicated().sum())
    duplicate_ids = 0
    if "ID_Unico" in data.columns:
        duplicate_ids = int(data["ID_Unico"].duplicated(keep=False).sum())

    spectral_numeric = numeric_data[spectral_columns]
    finite_spectral = spectral_numeric.replace([np.inf, -np.inf], np.nan)
    invalid_spectral_values = int(
        spectral_numeric.isin([np.inf, -np.inf]).sum().sum()
    )
    out_of_range_reflectance = int(
        ((finite_spectral < 0) | (finite_spectral > 1)).sum().sum()
    )

    available_targets = [
        column for column in TARGET_COLUMNS if column in data.columns
    ]
    target_coverage = pd.DataFrame({
        "atributo": available_targets,
        "n_validos": [int(data[column].notna().sum()) for column in available_targets],
        "percentual_valido": [
            round(float(data[column].notna().mean() * 100), 2)
            for column in available_targets
        ],
    })

    report = {
        "sensor": sensor_name,
        "arquivo": str(file_path),
        "codificacao": encoding_used,
        "linhas_brutas": len(raw_data),
        "colunas_brutas": len(raw_data.columns),
        "linhas_apos_qc": len(data.drop_duplicates()),
        "colunas_apos_qc": len(data.columns),
        "bandas_espectrais": len(spectral_columns),
        "primeiro_comprimento_onda": min(map(float, spectral_columns)) if spectral_columns else np.nan,
        "ultimo_comprimento_onda": max(map(float, spectral_columns)) if spectral_columns else np.nan,
        "valores_ausentes": int(data.isna().sum().sum()),
        "linhas_duplicadas": duplicate_rows,
        "ids_duplicados": duplicate_ids,
        "valores_infinitos_espectrais": invalid_spectral_values,
        "refletancias_fora_de_0_a_1": out_of_range_reflectance,
        "colunas_sem_valores": len(empty_columns),
        "colunas_espectrais_nao_numericas": len(non_numeric_counts),
    }

    # Mantém dados faltantes para decisão posterior, mas elimina duplicatas exatas.
    clean_data = data.replace([np.inf, -np.inf], np.nan).drop_duplicates().reset_index(drop=True)
    clean_data.to_csv(OUTPUT_DIR / f"{sensor_name.lower()}_qc.csv", index=False)
    target_coverage.to_csv(OUTPUT_DIR / f"{sensor_name.lower()}_cobertura_alvos.csv", index=False)

    return report, target_coverage, spectral_columns, non_numeric_counts


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
reports = []
coverage_tables = []

for sensor_name, file_path in FILES.items():
    report, coverage, spectral_columns, non_numeric_counts = import_and_qc(
        file_path, sensor_name
    )
    reports.append(report)
    coverage_tables.append(coverage.assign(sensor=sensor_name))
    if non_numeric_counts:
        print(f"Aviso {sensor_name}: bandas não numéricas: {non_numeric_counts}")

qc_report = pd.DataFrame(reports)
target_coverage = pd.concat(coverage_tables, ignore_index=True)
qc_report.to_csv(OUTPUT_DIR / "relatorio_qc.csv", index=False)
target_coverage.to_csv(OUTPUT_DIR / "cobertura_atributos.csv", index=False)

print("Relatório de controle de qualidade")
display(qc_report)
print("Cobertura dos atributos laboratoriais")
display(target_coverage)
print(f"\nArquivos processados salvos em: {OUTPUT_DIR.resolve()}")


Relatório de controle de qualidade


,sensor,arquivo,codificacao,linhas_brutas,colunas_brutas,linhas_apos_qc,colunas_apos_qc,bandas_espectrais,primeiro_comprimento_onda,ultimo_comprimento_onda,valores_ausentes,linhas_duplicadas,ids_duplicados,valores_infinitos_espectrais,refletancias_fora_de_0_a_1,colunas_sem_valores,colunas_espectrais_nao_numericas
0,MIR,C:\Users\PC\Desktop\RESET\raw_mir_data_Saturin...,utf-8,49945,923,49945,921,901,400.0,4000.0,2670194,0,0,0,40135,1,0
1,NIR,C:\Users\PC\Desktop\RESET\raw_nir_data_Saturin...,cp1252,40137,2173,40137,2172,2151,350.0,2500.0,2430897,0,0,0,1786,0,0


Cobertura dos atributos laboratoriais


,atributo,n_validos,percentual_valido,sensor
0,Sand_gkg,42181,84.45,MIR
1,Silt_gkg,42288,84.67,MIR
2,Clay_gkg,43804,87.70,MIR
3,Density_kgdm3,19822,39.69,MIR
4,C_gkg,44850,89.80,MIR
5,P_mgkg,15616,31.27,MIR
6,pH_H2O,44974,90.05,MIR
7,Sand_gkg,28571,71.18,NIR
8,Silt_gkg,28629,71.33,NIR
9,Clay_gkg,29249,72.87,NIR



Arquivos processados salvos em: C:\Users\PC\Downloads\Material_Final_Data_Science_Solos\Material_Final_Data_Science_Solos\notebooks\outputs_importacao_qc



Resultados principais:

- MIR: 49.945 linhas e 901 bandas espectrais.
- NIR: 40.137 linhas e 2.151 bandas espectrais.
- MIR importado com UTF-8.
- NIR importado automaticamente com CP1252.
- Foram verificadas ausências, duplicatas, IDs repetidos, valores infinitos, bandas não numéricas e valores espectrais fora da faixa de 0 a 1.
- As cópias processadas e os relatórios foram salvos em:

`notebooks/outputs_importacao_qc/`

Arquivos gerados:

- `mir_qc.csv`
- `nir_qc.csv`
- `relatorio_qc.csv`
- `cobertura_atributos.csv`
- `mir_cobertura_alvos.csv`
- `nir_cobertura_alvos.csv`

Completed: *Importar dados dos arquivos CSV* (1/3)

